# Retail Sales Revenue Prediction System
## Predictive Modelling Using Ordinary Least Squares (OLS) Linear Regression
**Coursework:** Third-Year B.Tech AI & ML - Predictive Modelling

This notebook demonstrates the end-to-end data science and machine learning workflow:
1. **Data Ingestion & Integrity Auditing**
2. **Exploratory Data Analysis (EDA) & Outlier Detection (IQR)**
3. **Predictive Feature Engineering (Leakage-Free)**
4. **Chronological Train-Test Partitioning (80/20)**
5. **Scikit-learn Pipeline Construction (ColumnTransformer + StandardScaler)**
6. **Model Training: Ordinary Least Squares Linear Regression**
7. **Generalization & Evaluation Metrics (MAE, MSE, RMSE, R², MAPE)**
8. **Coefficient Interpretation & Residual Analysis**

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.append('..')

from src.data_loader import load_dataset, get_dataset_summary
from src.preprocessing import detect_outliers_iqr, chronological_train_test_split
from src.feature_engineering import prepare_modeling_data
from src.train_models import run_full_training_pipeline
from src.prediction import RetailSalesPredictor

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print("Libraries loaded successfully!")

## 1. Load Dataset

In [ ]:
df = load_dataset('../data/retail_sales.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Statistical Summary & Outlier Detection (IQR)

In [ ]:
summary = get_dataset_summary(df)
print(f"Total Revenue: ${summary['total_revenue']:,.2f}")
print(f"Average Revenue per Order: ${summary['average_revenue']:.2f}")

outliers = detect_outliers_iqr(df['Sales'])
print("\nIQR Outlier Statistics:")
for k, v in outliers.items():
    print(f"  {k}: {v}")

## 3. Train Linear Regression & Evaluate Generalization

In [ ]:
results = run_full_training_pipeline(df)
print("Training Set Metrics:", results['train_metrics'])
print("Testing Set Metrics:", results['test_metrics'])

## 4. Top Learned Regression Weights (Beta Coefficients)

In [ ]:
lr_diag = results['lr_diagnostics']
print(f"Fitted Intercept (b0): ${lr_diag['intercept']:.2f}")
lr_diag['coefficients_df'].head(12)

## 5. Live Inference Prediction Test

In [ ]:
predictor = RetailSalesPredictor('../models')
test_input = {
    'order_date': '2026-11-20',
    'category': 'Furniture',
    'subcategory': 'Chairs',
    'segment': 'Corporate',
    'region': 'West',
    'ship_mode': 'Standard Class',
    'discount': 0.10
}
pred = predictor.predict(test_input)
print("Linear Regression Forecast:", pred['predicted_revenue'])
print("Expected Range:", f"${pred['lower_bound_estimate']} - ${pred['upper_bound_estimate']}")
print("Summary:", pred['summary'])